# exp3 R1 7 boundary segmentation

**Self-contained Colab notebook.** Run the setup cells once (top to bottom), then the experiment. The four shared modules are written to `/content` automatically, so you do NOT need to upload a `shared/` folder. Edit Drive paths only in the `config_paths.py` cell below.

> Runtime: set **Runtime -> Change runtime type -> GPU** for the inference and training experiments.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('Drive mounted; /content on path.')

In [ ]:
# scikit-image backs metrics.calc_ssim; guarantee it for every notebook
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])
print('scikit-image ready')

In [ ]:
%%writefile /content/config_paths.py
"""
Central path + constant configuration for all JBHI revision Colab experiments.

Derived verbatim from the author's existing notebooks:
  - COLAB_IEEE_FIGURES.ipynb      (paper baselines + EPHNOGRAM chunks + split)
  - cross_dataset_validation.ipynb (SensSmart-Tech + Trail-4 generator)

Every experiment script imports from this file so paths live in ONE place.
If your Drive layout differs, edit ONLY this file.
"""

import os
import torch

# ============================================================================
# >>> EDIT THIS BLOCK IF YOUR DRIVE MOVES <<<
# Updated 2026-07-14: the dataset is now a SHARED folder mounted at
# /content/drive/MyDrive/JBHI-Revision/October Thesis Dataset. If it moves again,
# change ONLY DRIVE_ROOT and OCT_DIR. NOTE: DATA is READ from OCT_DIR; everything
# the code WRITES (retrained checkpoints, outputs) goes under your own Drive
# (OUT_ROOT), so a read-only shared dataset still works.
# ============================================================================
DRIVE_ROOT = "/content/drive/MyDrive"
# Shared dataset root: contains "Refine-Chunked", "Best-epoch-trail-4", etc. (read-only OK).
OCT_DIR    = f"{DRIVE_ROOT}/JBHI-Revision/October Thesis Dataset"

# ----------------------------------------------------------------------------
# EPHNOGRAM data (training-domain test set = 453 segments, 3 subjects)
# .npz keys per chunk: 'ecg', 'pcg', 'subject_id', 'scenario'
#   ecg, pcg : float32 arrays, 64000 samples (8 s @ 8 kHz), RAW (not normalized)
#   subject_id : e.g. 'S009', 'S019', 'S023'
#   scenario   : free-text, e.g. 'rest', 'exercise bicycle stress test'
# ----------------------------------------------------------------------------
EPH_CHUNKS_DIR = f"{OCT_DIR}/Refine-Chunked"
# Subject split. keys: train/val/test_subjects. IF THIS FILE IS ABSENT the
# notebooks fall back to an arbitrary split, which gives the WRONG test set
# (the 2026-07-10 run hit this: 22 segments / 4 subjects instead of the paper's
# 453 segments / 3 subjects). Upload the real split JSON, or point this at it.
SPLIT_FILE     = f"{EPH_CHUNKS_DIR}/train_val_test_split.json"

# ----------------------------------------------------------------------------
# Model checkpoints  --  CONFIRM ALL FOUR EXIST IN THE NEW ACCOUNT
#
#   "Proposed" MUST be the Trail-4 generator (polarity fix, BatchNorm output),
#   i.e. trail4_best.pt or trail4_generator.pth. A "trial2_*" / Trail-2 file is
#   the WRONG (Tanh) model: it loads only partially under strict=False and
#   produces a near-flat, invalid reconstruction (see the 2026-07-10 run).
#
#   The three baselines lived under a SEPARATE top-level folder
#   (Paper1_ECG_Reconstruction) in the old account. Confirm they were migrated;
#   if they now sit under OCT_DIR, repoint them below.
# ----------------------------------------------------------------------------
CKPT = {
    # Baselines are RETRAINED by exp7 and SAVED to your own writable Drive
    # (DRIVE_ROOT/JBHI_Revision/baseline_checkpoints == OUT_ROOT/baseline_checkpoints),
    # so a read-only shared dataset is fine. Same paths are used to load them later.
    "CNN":      f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/CNN_EncoderDecoder/best_model.pt",
    "UNet":     f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/UNet_Baseline/best_model.pt",
    "WGAN-GP":  f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/WGAN_GP/best_model.pt",
    "Proposed": f"{OCT_DIR}/Best-epoch-trail-4/trail4_best.pt",   # read from the shared dataset
}
CKPT_PROPOSED_ALT = f"{OCT_DIR}/trail4_generator.pth"   # fallback copy (state_dict)

# ----------------------------------------------------------------------------
# SensSmart-Tech cross-dataset (32 subjects, 338 records @ 1 kHz -> 8 kHz)
#   WFDB pairs: {base}_ecg.hea/.dat and {base}_pcg.hea/.dat ; ECG lead II
# ----------------------------------------------------------------------------
SENSSMART_WFDB  = f"{OCT_DIR}/senssmarttech-database/WFDB"
SENSSMART_PREP  = f"{OCT_DIR}/milestone2_preprocessed"     # cached .npz (keys ecg/pcg)

# ----------------------------------------------------------------------------
# Output root for ALL revision artifacts (created on demand)
#   Each experiment writes to a named sub-folder here.
# ----------------------------------------------------------------------------
OUT_ROOT = f"{DRIVE_ROOT}/JBHI_Revision"

def out_dir(name):
    """Return (and create) an output sub-folder under OUT_ROOT."""
    p = os.path.join(OUT_ROOT, name)
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------------------------------------------------------
# Signal + inference constants
# ----------------------------------------------------------------------------
FS          = 8000        # sampling rate (Hz)
SEG_LEN     = 64000       # samples per segment (8 s)
NUM_SCENARIOS = 3         # rest / exercise / recovery one-hot
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# R-peak detector settings (identical to metrics.ipynb; keep fixed for reproducibility)
RPEAK_DISTANCE   = 400    # min inter-peak samples = 50 ms @ 8 kHz
RPEAK_PROMINENCE = 0.5

# SensSmart preprocessing bands (from cross_dataset_validation.ipynb)
ECG_BAND = (0.5, 40.0)
PCG_BAND = (25.0, 400.0)
FILTER_ORDER = 4

# Scenario one-hots
SCEN_REST     = [1.0, 0.0, 0.0]
SCEN_EXERCISE = [0.0, 1.0, 0.0]
SCEN_RECOVERY = [0.0, 0.0, 1.0]


def scenario_to_index(scenario_str):
    """Map EPHNOGRAM free-text scenario -> class index {0,1,2}.

    Matches the mapping used in COLAB_IEEE_FIGURES.ipynb.
    """
    s = str(scenario_str).lower()
    if "rest" in s or "laying" in s:
        return 0
    if "exercise" in s or "bicycle" in s or "treadmill" in s or "running" in s:
        return 1
    return 2

In [ ]:
%%writefile /content/models.py
"""
Model architectures for the JBHI revision experiments.

Reproduced VERBATIM from the author's COLAB_IEEE_FIGURES.ipynb (paper Table II
baselines) and models.ipynb (ablation trails), so the checkpoints in
config_paths.CKPT load cleanly.

Which class loads which checkpoint
----------------------------------
    CKPT['CNN']      -> CNNEncoderDecoder()        # no scenario, forward(x)
    CKPT['UNet']     -> UNetGenerator()            # pix2pix, forward(x)
    CKPT['WGAN-GP']  -> UNetGenerator()            # same generator as U-Net
    CKPT['Proposed'] -> ImprovedGeneratorNoTanh()  # scenario, forward(x, s_onehot)

The proposed model's forward takes a scenario one-hot; the three baselines take
only the PCG. Every inference script must branch on this (helper `run_generator`
below does it for you).

For the R1.4 component ablation, use `ConfigurableGenerator`, which exposes the
three grouped factors as switches: use_attention, use_dilated, and
output_activation in {'batchnorm','tanh','linear'}.

NOTE ON PARAM COUNTS: the submitted paper's numbers (CNN 9.8M, U-Net 13.6M,
Proposed 14.0M) do NOT match these classes (which give ~larger counts). Always
report count_params(model_loaded_from_checkpoint) computed here -- see R2.2/R2.5.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# Shared bottleneck modules
# ===========================================================================
class MultiHeadSelfAttention(nn.Module):
    """8-head self-attention over the temporal axis at the bottleneck."""
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)          # (3, B, heads, T, head_dim)
        # .contiguous() is REQUIRED for SDPA to select the flash / memory-efficient
        # kernel (O(T) memory). Without it the permuted views fall back to the MATH
        # kernel, which materializes the T x T score matrix -- that OOM-ed the
        # attention variants even on a 40 GB A100 at batch 48. Mathematically
        # identical to softmax(QK^T/sqrt(d))V, so existing checkpoints are unaffected.
        q, k, v = qkv[0].contiguous(), qkv[1].contiguous(), qkv[2].contiguous()
        p = self.dropout.p if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
        out = out.transpose(2, 3).reshape(B, C, T)
        return self.proj(out) + x


class MultiScaleDilatedBlock(nn.Module):
    """Parallel dilated convs at rates 1, 2, 4 with a 1x1 fusion + residual."""
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3
        self.conv_d1 = nn.Conv1d(channels, c1, 3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, 3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, 3, padding=4, dilation=4)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, 1),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        cat = torch.cat([self.conv_d1(x), self.conv_d2(x), self.conv_d4(x)], dim=1)
        return self.fusion(cat) + x


class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


# ===========================================================================
# Paper Table II baseline: CNN Encoder-Decoder (8-level, no skips, no scenario)
# ===========================================================================
class ConvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class CNNEncoderDecoder(nn.Module):
    """Paper CNN baseline -> loads CKPT['CNN']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        bf = base_filters
        self.enc1 = ConvBlock(in_channels, bf)
        self.enc2 = ConvBlock(bf, bf * 2)
        self.enc3 = ConvBlock(bf * 2, bf * 4)
        self.enc4 = ConvBlock(bf * 4, bf * 8)
        self.enc5 = ConvBlock(bf * 8, bf * 8)
        self.enc6 = ConvBlock(bf * 8, bf * 8)
        self.enc7 = ConvBlock(bf * 8, bf * 8)
        self.enc8 = ConvBlock(bf * 8, bf * 8)
        self.dec1 = DeconvBlock(bf * 8, bf * 8)
        self.dec2 = DeconvBlock(bf * 8, bf * 8)
        self.dec3 = DeconvBlock(bf * 8, bf * 8)
        self.dec4 = DeconvBlock(bf * 8, bf * 8)
        self.dec5 = DeconvBlock(bf * 8, bf * 4)
        self.dec6 = DeconvBlock(bf * 4, bf * 2)
        self.dec7 = DeconvBlock(bf * 2, bf)
        self.final = nn.ConvTranspose1d(bf, out_channels, 4, 2, 1)

    def forward(self, x):
        n = x.size(2)
        e = self.enc1(x); e = self.enc2(e); e = self.enc3(e); e = self.enc4(e)
        e = self.enc5(e); e = self.enc6(e); e = self.enc7(e); e = self.enc8(e)
        d = self.dec1(e); d = self.dec2(d); d = self.dec3(d); d = self.dec4(d)
        d = self.dec5(d); d = self.dec6(d); d = self.dec7(d)
        out = self.final(d)
        if out.size(2) != n:
            diff = n - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Paper Table II baseline: pix2pix U-Net (shared by U-Net and WGAN-GP)
# ===========================================================================
class UNetBlock(nn.Module):
    def __init__(self, i, o, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(i, o, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm1d(o))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, i, o, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose1d(i, o, 4, 2, 1, bias=False),
                  nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        if x.size(2) != skip.size(2):
            diff = skip.size(2) - x.size(2)
            x = F.pad(x, (diff // 2, diff - diff // 2))
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """Paper U-Net / WGAN-GP generator -> loads CKPT['UNet'] or CKPT['WGAN-GP']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super().__init__()
        f = features
        self.e1 = UNetBlock(in_channels, f, normalize=False)
        self.e2 = UNetBlock(f, f * 2); self.e3 = UNetBlock(f * 2, f * 4)
        self.e4 = UNetBlock(f * 4, f * 8); self.e5 = UNetBlock(f * 8, f * 8)
        self.e6 = UNetBlock(f * 8, f * 8); self.e7 = UNetBlock(f * 8, f * 8)
        self.e8 = UNetBlock(f * 8, f * 8, normalize=False)
        self.d1 = UNetUpBlock(f * 8, f * 8, dropout=0.5)
        self.d2 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d3 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d4 = UNetUpBlock(f * 16, f * 8)
        self.d5 = UNetUpBlock(f * 16, f * 4); self.d6 = UNetUpBlock(f * 8, f * 2)
        self.d7 = UNetUpBlock(f * 4, f)
        self.final = nn.ConvTranspose1d(f * 2, out_channels, 4, 2, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        d = self.d1(e8, e7); d = self.d2(d, e6); d = self.d3(d, e5); d = self.d4(d, e4)
        d = self.d5(d, e3); d = self.d6(d, e2); d = self.d7(d, e1)
        out = self.final(d)
        if out.size(2) != x.size(2):
            diff = x.size(2) - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Proposed generator (Trail-4): U-Net + MHSA + MSDC + scenario, BatchNorm output
# ===========================================================================
class ImprovedGeneratorNoTanh(nn.Module):
    """Proposed model -> loads CKPT['Proposed']. forward(x, scenario_onehot)."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1), nn.BatchNorm1d(1))

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        e1 = self.enc1(x) + s
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Configurable generator for the R1.4 component ablation
# ===========================================================================
class ConfigurableGenerator(nn.Module):
    """Ablation generator: toggle each grouped factor independently.

    use_attention : add MHSA in the 4 bottleneck blocks
    use_dilated   : add MSDC in the 4 bottleneck blocks
    output_activation : 'batchnorm' (proposed) | 'tanh' | 'linear'
    use_scenario  : condition on the scenario one-hot (keep True to match proposed)

    Ablation rows the reviewer asked for map to:
        U-Net only          -> use_attention=False, use_dilated=False
        U-Net + MSDC        -> use_attention=False, use_dilated=True
        U-Net + MHSA        -> use_attention=True,  use_dilated=False
        U-Net + both        -> use_attention=True,  use_dilated=True
        Proposed + Tanh     -> both True, output_activation='tanh'
        Proposed + linear   -> both True, output_activation='linear'
        Proposed (BatchNorm)-> both True, output_activation='batchnorm'
    Polarity-on/off is a DATA-loader switch; adversarial-on/off is a TRAINING
    switch -- neither is an architecture change, so control them in the trainer.
    """
    def __init__(self, base_channels=64, num_scenarios=3,
                 use_attention=True, use_dilated=True,
                 output_activation="batchnorm", use_scenario=True):
        super().__init__()
        bc = base_channels
        self.use_scenario = use_scenario
        if use_scenario:
            self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(*[
            EnhancedResidualBlock(bc*8, use_attention=use_attention, use_dilated=use_dilated)
            for _ in range(4)])
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        head = [nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1)]
        if output_activation == "batchnorm":
            head.append(nn.BatchNorm1d(1))
        elif output_activation == "tanh":
            head.append(nn.Tanh())
        elif output_activation == "linear":
            pass
        else:
            raise ValueError(f"unknown output_activation {output_activation}")
        self.dec1 = nn.Sequential(*head)

    def forward(self, x, scenario_onehot=None):
        e1 = self.enc1(x)
        if self.use_scenario and scenario_onehot is not None:
            e1 = e1 + self.scenario_embed(scenario_onehot).unsqueeze(2)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Discriminator (for R1.4 adversarial retraining)
# ===========================================================================
class SpectralNormDiscriminator(nn.Module):
    """Spectrally-normalized PatchGAN critic with scenario conditioning."""
    def __init__(self, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        sn = nn.utils.spectral_norm
        self.conv1 = sn(nn.Conv1d(2, bc, 7, 2, 3))
        self.conv2 = sn(nn.Conv1d(bc, bc*2, 5, 2, 2)); self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = sn(nn.Conv1d(bc*2, bc*4, 5, 2, 2)); self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = sn(nn.Conv1d(bc*4, bc*8, 5, 2, 2)); self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = sn(nn.Conv1d(bc*8, bc*8, 5, 2, 2)); self.bn5 = nn.BatchNorm1d(bc*8)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        return torch.sigmoid(self.fc(self.pool(x).squeeze(2)))


# ===========================================================================
# WGAN-GP critic (for retraining the WGAN-GP baseline; extracted verbatim from
# the original 05_GAN_WGAN_GP notebook). No BatchNorm, no sigmoid -> a valid
# WGAN critic. The generator is the plain UNetGenerator above.
# ===========================================================================
class WGANCritic(nn.Module):
    def __init__(self, in_channels=2, features=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(in_channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features, features * 2, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 2, features * 4, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 4, features * 8, 4, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 8, 1, 4, 1, 1),
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))


def compute_gradient_penalty(critic, pcg, real_ecg, fake_ecg, device):
    """WGAN-GP gradient penalty. Interpolates the ECG only; PCG conditions."""
    bs = real_ecg.size(0)
    alpha = torch.rand(bs, 1, 1, device=device)
    interp = (alpha * real_ecg + (1 - alpha) * fake_ecg).requires_grad_(True)
    c_interp = critic(pcg, interp)
    grads = torch.autograd.grad(
        outputs=c_interp, inputs=interp,
        grad_outputs=torch.ones_like(c_interp),
        create_graph=True, retain_graph=True)[0]
    grads = grads.view(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


# ===========================================================================
# Loading + utility helpers
# ===========================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_model(model, checkpoint_path, device="cpu", strict=False):
    """Load a checkpoint into `model`, handling common save formats."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        for key in ("generator_state_dict", "model_state_dict", "state_dict"):
            if key in ckpt:
                ckpt = ckpt[key]; break
    state = {k.replace("module.", ""): v for k, v in ckpt.items()} \
            if isinstance(ckpt, dict) else ckpt
    result = model.load_state_dict(state, strict=strict)
    # Loud guard: a large key mismatch means the checkpoint does not fit this
    # architecture (e.g. a Trail-2/trial2 file loaded into the Trail-4 model).
    n_model = max(len(model.state_dict()), 1)
    n_missing, n_unexpected = len(result.missing_keys), len(result.unexpected_keys)
    tag = os.path.basename(checkpoint_path)
    if n_missing or n_unexpected:
        flag = "  <-- LIKELY WRONG CHECKPOINT FOR THIS MODEL" if n_missing > 0.10 * n_model else ""
        print(f"[load_model] {tag}: {n_missing}/{n_model} params missing, "
              f"{n_unexpected} unexpected{flag}")
    else:
        print(f"[load_model] {tag}: exact match ({n_model} tensors).")
    model.to(device).eval()
    return model, result


def build_paper_models(ckpt_dict, device="cpu"):
    """Load all four paper models. Returns {name: model}."""
    models = {}
    models["CNN"], _     = load_model(CNNEncoderDecoder(), ckpt_dict["CNN"], device)
    models["UNet"], _    = load_model(UNetGenerator(), ckpt_dict["UNet"], device)
    models["WGAN-GP"], _ = load_model(UNetGenerator(), ckpt_dict["WGAN-GP"], device)
    models["Proposed"], _= load_model(ImprovedGeneratorNoTanh(), ckpt_dict["Proposed"], device)
    return models


def run_generator(model, pcg_tensor, scenario_tensor, is_proposed):
    """Uniform inference call. `is_proposed` selects the (x, s) signature."""
    with torch.no_grad():
        if is_proposed:
            out = model(pcg_tensor, scenario_tensor)
        else:
            out = model(pcg_tensor)
    return out.squeeze().cpu().numpy()

In [ ]:
%%writefile /content/metrics.py
"""
Metrics for the JBHI revision.

Two families:
  (A) Signal-fidelity metrics  -- identical to the submitted paper's metrics.ipynb
                                  (PSNR, SNR, SSIM, Pearson r, MAE, RMSE,
                                   R-peak AMPLITUDE error, QRS-width error).
  (B) Temporal R-peak DETECTION metrics  -- NEW, added to answer Reviewer 1 #1.
       These detect R-peaks INDEPENDENTLY in the reconstructed ECG and match
       them to ground-truth peaks within a tolerance window, then report
       sensitivity, PPV, F1, FP, FN, timing-error stats, RR-interval error,
       and heart-rate error. This is the metric the paper was missing.

Keep the detector settings fixed (config_paths.RPEAK_*) so every experiment and
the paper agree.
"""

import numpy as np
import scipy.signal as signal
from scipy.stats import pearsonr


# ---------------------------------------------------------------------------
# R-peak detection (single, shared definition)
# ---------------------------------------------------------------------------
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """scipy find_peaks on an ECG trace. Returns integer sample indices."""
    peaks, _ = signal.find_peaks(np.asarray(ecg).ravel(),
                                 distance=distance, prominence=prominence)
    return peaks


# ---------------------------------------------------------------------------
# (A) Signal-fidelity metrics
# ---------------------------------------------------------------------------
def calc_psnr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_snr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    noise = real - fake
    return float(10 * np.log10(np.sum(real ** 2) / (np.sum(noise ** 2) + 1e-12)))


def calc_correlation(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    try:
        c, _ = pearsonr(real, fake)
        return float(c)
    except Exception:
        return 0.0


def calc_ssim(real, fake):
    from skimage.metrics import structural_similarity as compare_ssim  # lazy: Colab-only dep
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    r = (real - real.min()) / (real.max() - real.min() + 1e-8)
    f = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(compare_ssim(r, f, data_range=1.0))


def calc_mae(real, fake):
    return float(np.mean(np.abs(np.asarray(real).ravel() - np.asarray(fake).ravel())))


def calc_rmse(real, fake):
    d = np.asarray(real).ravel() - np.asarray(fake).ravel()
    return float(np.sqrt(np.mean(d ** 2)))


def calc_r_peak_amplitude_error(real, fake, fs=8000, normalize01=True, **kw):
    """Paper's 'R-peak error' -> renamed 'R-peak AMPLITUDE error'.

    Detects R-peaks on the GROUND TRUTH only, then averages the amplitude error at
    those locations. Kept ONLY as a fidelity measure; NOT a timing metric (R1 #1).

    normalize01=True (DEFAULT) min-max normalizes both signals to [0,1] before the
    error, which REPRODUCES the paper's ~0.126 scale (confirmed: the paper's metric
    is [0,1]-normalized). normalize01=False gives the raw z-scored form, which is
    ~8x larger (z-scored ECG spans ~6-8 units vs 1 for [0,1]) -- that is what the
    first exp4/6/7 runs reported (~1.0). Use the default for consistency with the
    manuscript. Returns np.nan when the ground truth has no detectable R-peaks.
    """
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    if normalize01:
        real = (real - real.min()) / (real.max() - real.min() + 1e-8)
        fake = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5, search_ms=25):
    amp = abs(ecg[peak_idx]); thr = threshold * amp
    win = int(search_ms / 1000 * fs)
    q = peak_idx
    for j in range(peak_idx - 1, max(peak_idx - win, 0), -1):
        if abs(ecg[j]) < thr:
            q = j; break
    s = peak_idx
    for j in range(peak_idx + 1, min(peak_idx + win, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j; break
    return (s - q) / fs * 1000.0


def calc_qrs_width_error(real, fake, fs=8000, **kw):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    errs = [abs(_qrs_width_at_peak(real, p, fs) - _qrs_width_at_peak(fake, p, fs))
            for p in peaks]
    return float(np.mean(errs))


# ---------------------------------------------------------------------------
# (B) Temporal R-peak DETECTION metrics  (NEW -- Reviewer 1 #1)
# ---------------------------------------------------------------------------
def match_peaks(true_peaks, pred_peaks, tol_samples):
    """Greedy nearest-neighbour matching within a tolerance window.

    Each true peak claims the closest unused predicted peak within tol_samples.
    Returns (matches, TP, FP, FN, signed_timing_errors_samples).
    signed error = pred_index - true_index (positive = reconstructed peak late).
    """
    true_peaks = list(np.asarray(true_peaks).astype(int))
    pred_peaks = list(np.asarray(pred_peaks).astype(int))
    used, matches, terr = set(), [], []
    for tp in true_peaks:
        cands = [(abs(pp - tp), pp) for pp in pred_peaks
                 if pp not in used and abs(pp - tp) <= tol_samples]
        if cands:
            _, pp = min(cands)
            used.add(pp); matches.append((tp, pp)); terr.append(pp - tp)
    TP = len(matches)
    FN = len(true_peaks) - TP
    FP = len(pred_peaks) - TP
    return matches, TP, FP, FN, np.asarray(terr, dtype=float)


def temporal_rpeak_metrics(true_ecg, pred_ecg, fs=8000, tol_ms=50,
                           distance=400, prominence=0.5):
    """Full temporal detection report for ONE segment.

    Returns a dict with detection (Se/PPV/F1/FP/FN), timing-error (ms),
    RR-interval error (ms), and heart-rate error (bpm). Segments with no
    ground-truth peaks return n_true=0 and NaN rates (exclude from means, but
    COUNT them -- a reconstruction that yields peaks where truth has none is a
    real failure; see caller handling).
    """
    tol = int(round(tol_ms / 1000 * fs))
    t_pk = detect_r_peaks(true_ecg, fs, distance, prominence)
    p_pk = detect_r_peaks(pred_ecg, fs, distance, prominence)

    matches, TP, FP, FN, terr = match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)

    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t   = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t   = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t   = float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25)) \
              if terr_ms.size else np.nan

    # RR-interval error on consecutive MATCHED pairs only (avoids missed-beat bias)
    rr_err = np.nan
    if len(matches) >= 2:
        tp_sorted = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(tp_sorted[:-1], tp_sorted[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    # Heart-rate error (bpm), estimator = 60000 / median-RR
    def _hr(pk):
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)
    hr_true, hr_pred = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_true - hr_pred)) if not (np.isnan(hr_true) or np.isnan(hr_pred)) else np.nan

    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_true, "hr_pred_bpm": hr_pred, "hr_error_bpm": hr_err,
    }


def aggregate_temporal(rows):
    """Aggregate a list of per-segment temporal_rpeak_metrics dicts.

    Detection totals (TP/FP/FN) are POOLED then Se/PPV/F1 recomputed (the
    correct way to aggregate detection over segments). Timing/RR/HR errors are
    averaged over segments where they are defined; the valid-N is reported so
    the denominator is transparent.
    """
    rows = [r for r in rows if r is not None]
    TP = sum(r["TP"] for r in rows)
    FP = sum(r["FP"] for r in rows)
    FN = sum(r["FN"] for r in rows)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = 2 * sens * ppv / (sens + ppv) if (sens + ppv) else np.nan

    def _mean(key):
        v = [r[key] for r in rows if r[key] is not None and not np.isnan(r[key])]
        return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)

    out = {"n_segments": len(rows), "TP": TP, "FP": FP, "FN": FN,
           "sensitivity": sens, "ppv": ppv, "f1": f1}
    for k in ["mae_timing_ms", "median_timing_ms", "iqr_timing_ms",
              "rr_error_ms", "hr_error_bpm"]:
        m, s, n = _mean(k)
        out[k + "_mean"], out[k + "_std"], out[k + "_validN"] = m, s, n
    return out


# ---------------------------------------------------------------------------
# Preprocessing helpers (shared with the paper pipeline)
# ---------------------------------------------------------------------------
def ensure_positive_polarity(ecg):
    """Flip so R-peaks point up. Applied to the ECG TARGET only (never PCG)."""
    ecg = np.asarray(ecg)
    return -ecg if np.abs(ecg.min()) > np.abs(ecg.max()) else ecg


def normalize_zscore(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return ((sig - sig.mean()) / (sig.std() + 1e-8)).astype(np.float32)

In [ ]:
%%writefile /content/ieee_style.py
"""
IEEE double-column figure style for the JBHI revision.

Design goals (per IEEE Transactions / J-BHI figure rules):
  * Vector output (PDF) + high-DPI PNG (600) for every figure.
  * Serif face (Times) to match IEEE body text; DejaVu Serif fallback on Colab.
  * BOLD axis labels that always carry explicit units.
  * Thick spines and ticks so lines stay legible at column scale.
  * constrained_layout everywhere -> no text/label/legend overlaps.
  * Column widths fixed to IEEE spec: single = 3.5 in, double = 7.16 in.

Usage
-----
    from ieee_style import set_ieee_style, new_fig, finalize_axes, save_fig
    set_ieee_style()
    fig, ax = new_fig("single", height=2.2)
    ax.plot(t, y)
    finalize_axes(ax, xlabel="Time (s)", ylabel="Amplitude (a.u.)")
    save_fig(fig, out_dir, "fig_name")   # writes fig_name.pdf AND fig_name.png
"""

import matplotlib
matplotlib.use("Agg")            # headless-safe; remove if you want inline display
import matplotlib.pyplot as plt

# IEEE column widths (inches)
COL_SINGLE = 3.5
COL_DOUBLE = 7.16

# Colour-blind-safe palette (Okabe-Ito), stable role -> colour mapping
COLORS = {
    "truth":    "#009E73",   # bluish green  -> ground truth
    "proposed": "#0072B2",   # blue          -> proposed model
    "cnn":      "#D55E00",   # vermillion    -> CNN
    "unet":     "#CC79A7",   # reddish purple-> U-Net
    "wgan":     "#E69F00",   # orange        -> WGAN-GP
    "pcg":      "#666666",   # grey          -> PCG input
    "accent":   "#0072B2",
    "muted":    "#999999",
}


def set_ieee_style():
    """Apply the global rcParams. Call once at the top of every script."""
    plt.rcParams.update({
        "figure.dpi":        150,
        "savefig.dpi":       600,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.02,
        "figure.constrained_layout.use": True,
        "font.size":         8,
        "font.family":       "serif",
        "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset":  "dejavuserif",
        "text.usetex":       False,
        "axes.labelsize":    9,
        "axes.labelweight":  "bold",
        "axes.titlesize":    9,
        "axes.titleweight":  "bold",
        "axes.linewidth":    1.1,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.4,
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size":  3.5,
        "ytick.major.size":  3.5,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "legend.fontsize":   7.5,
        "legend.framealpha": 0.92,
        "legend.edgecolor":  "0.7",
        "legend.handlelength": 1.6,
        "lines.linewidth":   1.0,
        "lines.solid_capstyle": "round",
    })


def new_fig(width="single", height=2.2, nrows=1, ncols=1, **kw):
    """Create a figure sized to an IEEE column.

    width : 'single' (3.5 in), 'double' (7.16 in), or a float in inches.
    height: figure height in inches.
    Returns (fig, ax) or (fig, axes) exactly like plt.subplots.
    """
    w = {"single": COL_SINGLE, "double": COL_DOUBLE}.get(width, width)
    fig, ax = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
    return fig, ax


def finalize_axes(ax, xlabel=None, ylabel=None, title=None,
                  legend=False, legend_loc="best", legend_ncol=1):
    """Apply bold labels with units, thicken spines, tidy ticks.

    Always pass xlabel/ylabel WITH units, e.g. 'Time (ms)', 'PSNR (dB)'.
    """
    if xlabel:
        ax.set_xlabel(xlabel, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontweight="bold")
    if title:
        ax.set_title(title, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
    ax.tick_params(axis="both", which="major", width=1.0, labelsize=8)
    if legend:
        leg = ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=True)
        leg.get_frame().set_linewidth(0.6)
    return ax


def save_fig(fig, out_directory, name):
    """Save both a vector PDF (for LaTeX) and a 600-dpi PNG (for preview)."""
    import os
    pdf = os.path.join(out_directory, f"{name}.pdf")
    png = os.path.join(out_directory, f"{name}.png")
    fig.savefig(pdf)
    fig.savefig(png)
    plt.close(fig)
    print(f"  saved  {pdf}")
    print(f"  saved  {png}")
    return pdf, png

In [ ]:
import importlib
import config_paths, models, metrics, ieee_style
for m in (config_paths, models, metrics, ieee_style):
    importlib.reload(m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

## Experiment

# exp3 - EPHNOGRAM three-arm cardiac-cycle boundary study (Reviewer 1, comment R1.7)

This experiment delivers the same-dataset true-versus-reconstructed boundary
comparison the reviewer requested. EPHNOGRAM supplies paired ECG for every test
segment, so a true-ECG gold reference is available on the same signals.

For each EPHNOGRAM test segment we derive cardiac-cycle phase boundaries from
R-peaks obtained three ways.

* Arm 1 (gold): R-peaks from the **true ECG**.
* Arm 2: R-peaks from the **reconstructed ECG** (Proposed generator).
* Arm 3: S1 onsets from **PCG only** (Shannon-energy envelope).

The fixed-fraction phase model splits each cardiac cycle into S1 (0-15%),
systole (15-50%), S2 (50-60%), and diastole (60-100%) of the cycle length.
We report the mean absolute boundary error (ms) and the S1 and S2 onset
localization error (ms) of arm 2 and arm 3 against arm 1.

Outputs land in `config_paths.out_dir('exp3_boundary')`:
a JSON of every number, a paste-ready LaTeX table, and one IEEE double-column
figure (PDF plus 600 dpi PNG).

**Reuse note.** This script reuses the Proposed reconstructions saved by exp1
under `out_dir('exp1_reconstructions')`. If that cache is absent, this script
regenerates the reconstructions with the Proposed checkpoint and writes the
same cache, so the two experiments stay consistent.

## Cell 1 - environment, Drive mount, shared imports

The four shared modules live in `JBHI_Revision/shared` on Drive. Copy the local
`Colab_Experiments/shared/` folder there once, or paste the modules into Colab.

In [ ]:
# Mount Drive and expose the shared modules.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not in Colab. Assuming shared modules are already importable.')

import sys
SHARED_DIR = '/content/drive/MyDrive/JBHI_Revision/shared'
if SHARED_DIR not in sys.path:
    sys.path.append(SHARED_DIR)

print('NOTE: copy the local Colab_Experiments/shared/ folder to')
print('      /content/drive/MyDrive/JBHI_Revision/shared  (or paste the modules in)')
print('      before running. All paths and helpers come from those modules.')

import os
import json
import glob
import numpy as np
import torch
import scipy.signal as sp_signal
from scipy.stats import pearsonr

import config_paths as CFG
import models as MDL
import metrics as MET
import ieee_style as IEEE

IEEE.set_ieee_style()
DEVICE = CFG.DEVICE
print(f'Device: {DEVICE}')

## Cell 2 - phase model, tolerances, output folder

The phase fractions and R-peak detector settings match the paper. The output
folder is created on demand through `config_paths.out_dir`.

In [ ]:
# Fixed-fraction cardiac-cycle phase model (start of each phase, as a cycle fraction).
PHASE_FRACTIONS = {
    'S1':       0.00,   # S1 onset  = cycle anchor
    'systole':  0.15,   # systole onset
    'S2':       0.50,   # S2 onset
    'diastole': 0.60,   # diastole onset
}
PHASE_ORDER = ['S1', 'systole', 'S2', 'diastole']

FS = CFG.FS                       # 8000 Hz
RPEAK_DISTANCE = CFG.RPEAK_DISTANCE
RPEAK_PROMINENCE = CFG.RPEAK_PROMINENCE
MATCH_TOL_MS = 150.0              # anchor-to-gold matching window (ms)

OUT_DIR = CFG.out_dir('exp3_boundary')
RECON_DIR = CFG.out_dir('exp1_reconstructions')          # shared with exp1
RECON_BUNDLE = os.path.join(RECON_DIR, 'proposed_reconstructions.npz')

# Preferred example segment for the figure (falls back to best correlation if absent).
PREFERRED_EXAMPLE = 'ECGPCG0001_chunk181'

print(f'Output folder:        {OUT_DIR}')
print(f'Reconstruction cache: {RECON_BUNDLE}')

## Cell 3 - PCG-only S1 detection (Shannon-energy envelope)

Arm 3 uses PCG alone. This detector isolates the heart-sound band, computes the
average Shannon energy envelope, picks heart-sound peaks, then labels S1 with
the systole-shorter-than-diastole duration asymmetry. No ECG is used.

In [ ]:
def _bandpass(sig, fs, lo, hi, order=4):
    """Zero-phase Butterworth bandpass. Returns the input unchanged on failure."""
    try:
        nyq = fs / 2.0
        b, a = sp_signal.butter(order, [lo / nyq, hi / nyq], btype='band')
        return sp_signal.filtfilt(b, a, sig).astype(np.float32)
    except Exception:
        return np.asarray(sig, dtype=np.float32)


def shannon_envelope(pcg, fs=FS, band=(25.0, 400.0), smooth_s=0.02):
    """Average Shannon energy envelope of a PCG trace, z-scored to unit variance."""
    x = np.asarray(pcg, dtype=np.float32).ravel()
    x = x / (np.max(np.abs(x)) + 1e-8)
    xf = _bandpass(x, fs, band[0], band[1])
    xf = xf / (np.max(np.abs(xf)) + 1e-8)
    eps = 1e-8
    energy = -(xf ** 2) * np.log(xf ** 2 + eps)
    win = max(3, int(round(smooth_s * fs)))
    kernel = np.hanning(win)
    kernel = kernel / (kernel.sum() + eps)
    env = np.convolve(energy, kernel, mode='same')
    env = (env - env.mean()) / (env.std() + eps)
    return env.astype(np.float32)


def detect_s1_peaks(pcg, fs=FS, min_gap_s=0.08, prominence=0.20):
    """Detect S1 onsets from PCG only. Returns sorted integer sample indices."""
    env = shannon_envelope(pcg, fs)
    dist = max(1, int(round(min_gap_s * fs)))
    peaks, _ = sp_signal.find_peaks(env, distance=dist, prominence=prominence)
    if len(peaks) < 3:
        return peaks.astype(int)

    intervals = np.diff(peaks).astype(float)
    even_int = intervals[0::2]     # intervals that follow peaks 0, 2, 4, ...
    odd_int = intervals[1::2]      # intervals that follow peaks 1, 3, 5, ...
    mean_even = float(np.mean(even_int)) if even_int.size else np.inf
    mean_odd = float(np.mean(odd_int)) if odd_int.size else np.inf
    overall = float(np.mean(intervals))

    if abs(mean_even - mean_odd) < 0.10 * overall:
        # Ambiguous durations. Fall back to the louder peak set as S1.
        amp_even = float(np.mean(env[peaks[0::2]])) if peaks[0::2].size else -np.inf
        amp_odd = float(np.mean(env[peaks[1::2]])) if peaks[1::2].size else -np.inf
        s1 = peaks[0::2] if amp_even >= amp_odd else peaks[1::2]
    else:
        # Systole is the shorter interval. Its starting peak is S1.
        s1 = peaks[0::2] if mean_even <= mean_odd else peaks[1::2]
    return np.sort(np.asarray(s1, dtype=int))

## Cell 4 - phase boundaries and boundary error

Each anchor plus its following anchor defines one cycle. The phase boundaries
sit at fixed fractions of that cycle. Boundary error is the absolute time gap
(ms) between an arm boundary and the matched gold boundary.

In [ ]:
def _next_interval_map(anchors):
    """Map each anchor (except the last) to its cycle length in samples."""
    anchors = np.sort(np.asarray(anchors, dtype=int))
    return {int(anchors[i]): int(anchors[i + 1] - anchors[i])
            for i in range(len(anchors) - 1)}


def boundary_errors_vs_gold(arm_anchors, gold_anchors, fs=FS, tol_ms=MATCH_TOL_MS):
    """Per-phase absolute boundary errors (ms) of an arm against the gold arm.

    Returns a dict {phase: [errors_ms]} plus the number of matched cycles.
    """
    out = {p: [] for p in PHASE_ORDER}
    gold_anchors = np.sort(np.asarray(gold_anchors, dtype=int))
    arm_anchors = np.sort(np.asarray(arm_anchors, dtype=int))
    if len(gold_anchors) < 2 or len(arm_anchors) < 2:
        return out, 0

    gold_len = _next_interval_map(gold_anchors)
    arm_len = _next_interval_map(arm_anchors)
    tol = int(round(tol_ms / 1000.0 * fs))

    matches, _, _, _, _ = MET.match_peaks(gold_anchors, arm_anchors, tol)
    n_cycles = 0
    for g, a in matches:
        lg = gold_len.get(int(g))
        la = arm_len.get(int(a))
        if lg is None or la is None:
            continue
        n_cycles += 1
        for phase in PHASE_ORDER:
            f = PHASE_FRACTIONS[phase]
            gold_b = g + f * lg
            arm_b = a + f * la
            out[phase].append(abs(arm_b - gold_b) / fs * 1000.0)
    return out, n_cycles

## Cell 5 - load the EPHNOGRAM test set and the Proposed reconstructions

This cell enumerates the test chunks, applies the paper preprocessing, and
obtains each reconstructed ECG from the exp1 cache or the Proposed checkpoint.
The reconstruction is rescaled to the true-ECG standard deviation, exactly as
the paper pipeline does on this same dataset.

In [ ]:
def _load_split_test_subjects():
    with open(CFG.SPLIT_FILE, 'r') as fh:
        splits = json.load(fh)
    for key in ('test_subjects', 'test'):
        if key in splits:
            return set(str(s) for s in splits[key])
    raise KeyError('No test-subject key found in split file.')


def _list_test_chunks():
    all_chunks = sorted(glob.glob(os.path.join(CFG.EPH_CHUNKS_DIR, '*.npz')))
    all_chunks = [p for p in all_chunks
                  if os.path.basename(p) != os.path.basename(CFG.SPLIT_FILE)]
    test_subjects = _load_split_test_subjects()
    test_chunks = []
    for path in all_chunks:
        try:
            data = np.load(path, allow_pickle=True)
        except Exception:
            continue
        if 'subject_id' not in data:
            continue
        if str(data['subject_id']) in test_subjects:
            test_chunks.append(path)
    return test_chunks, test_subjects


def _load_recon_cache():
    if os.path.exists(RECON_BUNDLE):
        try:
            npz = np.load(RECON_BUNDLE, allow_pickle=True)
            cache = {k: npz[k].astype(np.float32) for k in npz.files}
            print(f'Loaded {len(cache)} cached reconstructions from exp1.')
            return cache
        except Exception as exc:
            print(f'Could not read reconstruction cache: {exc}. Will regenerate.')
    else:
        print('No exp1 reconstruction cache found. Will regenerate as needed.')
    return {}


_PROPOSED = {'model': None}


def _get_proposed_model():
    """Lazily load the Proposed generator (only when regeneration is needed)."""
    if _PROPOSED['model'] is not None:
        return _PROPOSED['model']
    ckpt = CFG.CKPT['Proposed']
    if not os.path.exists(ckpt):
        ckpt = CFG.CKPT_PROPOSED_ALT
    if not os.path.exists(ckpt):
        raise FileNotFoundError('Proposed checkpoint not found at either path.')
    gen, _ = MDL.load_model(MDL.ImprovedGeneratorNoTanh(), ckpt, device=DEVICE)
    _PROPOSED['model'] = gen
    print(f'Loaded Proposed generator from {ckpt}')
    return gen


def _reconstruct_raw(pcg_norm, scenario_idx):
    """Run the Proposed generator on one z-scored PCG. Returns the raw ECG output."""
    gen = _get_proposed_model()
    pcg_t = torch.from_numpy(pcg_norm).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    scen_t = torch.zeros(1, CFG.NUM_SCENARIOS, device=DEVICE)
    scen_t[0, scenario_idx] = 1.0
    return MDL.run_generator(gen, pcg_t, scen_t, is_proposed=True).astype(np.float32)


# Build the per-segment records.
records = []
data_ok = os.path.exists(CFG.EPH_CHUNKS_DIR) and os.path.exists(CFG.SPLIT_FILE)
if not data_ok:
    print('EPHNOGRAM data or split file missing. Skipping. Check config_paths.')
else:
    test_chunks, test_subjects = _list_test_chunks()
    print(f'Test subjects: {sorted(test_subjects)}')
    print(f'Total test chunks: {len(test_chunks)}')

    recon_cache = _load_recon_cache()
    cache_dirty = False

    for idx, path in enumerate(test_chunks):
        key = os.path.splitext(os.path.basename(path))[0]
        data = np.load(path, allow_pickle=True)
        ecg_raw = np.asarray(data['ecg'], dtype=np.float32)
        pcg_raw = np.asarray(data['pcg'], dtype=np.float32)
        subject = str(data['subject_id'])
        scenario_str = str(data['scenario'])
        scenario_idx = CFG.scenario_to_index(scenario_str)

        # Paper preprocessing: polarity fix on ECG only, then z-score both.
        ecg_norm = MET.normalize_zscore(MET.ensure_positive_polarity(ecg_raw))
        pcg_norm = MET.normalize_zscore(pcg_raw)

        if key in recon_cache:
            recon_raw = recon_cache[key]
        else:
            recon_raw = _reconstruct_raw(pcg_norm, scenario_idx)
            recon_cache[key] = recon_raw
            cache_dirty = True

        # Rescale the reconstruction to the true-ECG std (same-dataset pipeline step).
        real_std = float(np.std(ecg_norm))
        fake_std = float(np.std(recon_raw))
        recon_scaled = (recon_raw * (real_std / (fake_std + 1e-8))).astype(np.float32)

        records.append({
            'key': key,
            'subject': subject,
            'scenario': scenario_str,
            'path': path,
            'ecg_norm': ecg_norm,
            'pcg_norm': pcg_norm,
            'recon_scaled': recon_scaled,
        })

        if (idx + 1) % 50 == 0:
            print(f'  prepared {idx + 1}/{len(test_chunks)} segments')

    if cache_dirty:
        try:
            np.savez_compressed(RECON_BUNDLE, **recon_cache)
            print(f'Saved reconstruction cache -> {RECON_BUNDLE}')
        except Exception as exc:
            print(f'Could not save reconstruction cache: {exc}')

    print(f'Prepared {len(records)} test segments.')

## Cell 6 - run the three-arm boundary analysis

This cell detects anchors for the three arms, computes boundary errors of arm 2
and arm 3 against the gold arm, then aggregates over every matched cycle.

In [ ]:
def _aggregate(pooled):
    """Summarize a per-phase list-of-errors dict into means, std, and pooled mean."""
    summary = {}
    all_errs = []
    for phase in PHASE_ORDER:
        vals = np.asarray(pooled[phase], dtype=float)
        all_errs.extend(vals.tolist())
        if vals.size:
            summary[f'{phase}_err_ms_mean'] = float(np.mean(vals))
            summary[f'{phase}_err_ms_std'] = float(np.std(vals))
        else:
            summary[f'{phase}_err_ms_mean'] = float('nan')
            summary[f'{phase}_err_ms_std'] = float('nan')
    all_errs = np.asarray(all_errs, dtype=float)
    summary['mean_abs_boundary_err_ms'] = float(np.mean(all_errs)) if all_errs.size else float('nan')
    summary['boundary_err_ms_std'] = float(np.std(all_errs)) if all_errs.size else float('nan')
    summary['n_matched_cycles'] = len(pooled[PHASE_ORDER[0]])
    return summary


results = {}
if records:
    pooled_recon = {p: [] for p in PHASE_ORDER}
    pooled_pcg = {p: [] for p in PHASE_ORDER}
    n_with_gold = 0

    for rec in records:
        gold_anchors = MET.detect_r_peaks(
            rec['ecg_norm'], fs=FS, distance=RPEAK_DISTANCE, prominence=RPEAK_PROMINENCE)
        if len(gold_anchors) < 2:
            rec['n_gold_cycles'] = 0
            continue
        n_with_gold += 1

        recon_anchors = MET.detect_r_peaks(
            rec['recon_scaled'], fs=FS, distance=RPEAK_DISTANCE, prominence=RPEAK_PROMINENCE)
        s1_anchors = detect_s1_peaks(rec['pcg_norm'], fs=FS)

        err_recon, n_r = boundary_errors_vs_gold(recon_anchors, gold_anchors)
        err_pcg, n_p = boundary_errors_vs_gold(s1_anchors, gold_anchors)
        for phase in PHASE_ORDER:
            pooled_recon[phase].extend(err_recon[phase])
            pooled_pcg[phase].extend(err_pcg[phase])

        # Keep light per-segment fields for figure selection.
        rec['n_gold_cycles'] = int(len(gold_anchors) - 1)
        rec['gold_anchors'] = gold_anchors
        rec['recon_anchors'] = recon_anchors
        rec['s1_anchors'] = s1_anchors
        try:
            rec['corr'] = float(pearsonr(rec['ecg_norm'], rec['recon_scaled'])[0])
        except Exception:
            rec['corr'] = float('nan')
        rec['recon_cycle_err_mean'] = (
            float(np.mean([v for p in PHASE_ORDER for v in err_recon[p]]))
            if n_r else float('nan'))

    results = {
        'description': ('Same-dataset true-versus-reconstructed cardiac-cycle boundary '
                        'study on the EPHNOGRAM test set. EPHNOGRAM provides paired ECG, '
                        'so arm 1 (true ECG) is a genuine gold reference on the same signals.'),
        'phase_model_fractions': PHASE_FRACTIONS,
        'fs_hz': FS,
        'match_tolerance_ms': MATCH_TOL_MS,
        'rpeak_distance_samples': RPEAK_DISTANCE,
        'rpeak_prominence': RPEAK_PROMINENCE,
        'n_test_segments': len(records),
        'n_segments_with_gold': n_with_gold,
        'arms': {
            'arm2_reconstructed_ecg': _aggregate(pooled_recon),
            'arm3_pcg_only_s1': _aggregate(pooled_pcg),
        },
    }

    json_path = os.path.join(OUT_DIR, 'exp3_boundary_results.json')
    with open(json_path, 'w') as fh:
        json.dump(results, fh, indent=2)
    print(f'Saved results JSON -> {json_path}')

    a2 = results['arms']['arm2_reconstructed_ecg']
    a3 = results['arms']['arm3_pcg_only_s1']
    print('\nMean absolute boundary error (ms):')
    print(f"  Reconstructed ECG : {a2['mean_abs_boundary_err_ms']:.2f}  "
          f"(cycles={a2['n_matched_cycles']})")
    print(f"  PCG-only S1       : {a3['mean_abs_boundary_err_ms']:.2f}  "
          f"(cycles={a3['n_matched_cycles']})")
else:
    print('No records available. Skipping analysis.')

## Cell 7 - paste-ready LaTeX table

The table lists the S1 onset error, S2 onset error, systole-boundary error,
diastole-boundary error, and mean absolute boundary error for each arm.

In [ ]:
def _fmt(x):
    return f'{x:.1f}' if (x == x) else '--'   # x != x guards NaN


if results:
    a2 = results['arms']['arm2_reconstructed_ecg']
    a3 = results['arms']['arm3_pcg_only_s1']

    lines = []
    lines.append('% ===================================================================')
    lines.append('% exp3 - EPHNOGRAM three-arm cardiac-cycle boundary error (R1.7)')
    lines.append('% Same-dataset true-versus-reconstructed comparison (paired ECG).')
    lines.append('% Paste into the Results section. Requires \\usepackage{booktabs}.')
    lines.append('% ===================================================================')
    lines.append('\\begin{table}[!t]')
    lines.append('\\centering')
    lines.append('\\caption{Cardiac-cycle phase-boundary error on the EPHNOGRAM test set. '
                 'Arm 1 (true ECG) is the gold reference. Errors are the mean absolute '
                 'time gap between each arm boundary and the gold boundary, in\\,ms. '
                 'The reconstructed-ECG arm recovers R-peak timing, so its boundaries '
                 'track the gold cycle. The PCG-only arm anchors on S1, so it lags by '
                 'the electromechanical delay.}')
    lines.append('\\label{tab:exp3_boundary}')
    lines.append('\\begin{tabular}{lccccc}')
    lines.append('\\toprule')
    lines.append('Arm & S1 onset & S2 onset & Systole & Diastole & Mean \\\\')
    lines.append(' & (ms) & (ms) & (ms) & (ms) & (ms) \\\\')
    lines.append('\\midrule')
    lines.append('True ECG (gold) & 0.0 & 0.0 & 0.0 & 0.0 & 0.0 \\\\')
    lines.append('Reconstructed ECG & '
                 f"{_fmt(a2['S1_err_ms_mean'])} & {_fmt(a2['S2_err_ms_mean'])} & "
                 f"{_fmt(a2['systole_err_ms_mean'])} & {_fmt(a2['diastole_err_ms_mean'])} & "
                 f"{_fmt(a2['mean_abs_boundary_err_ms'])} \\\\")
    lines.append('PCG-only S1 & '
                 f"{_fmt(a3['S1_err_ms_mean'])} & {_fmt(a3['S2_err_ms_mean'])} & "
                 f"{_fmt(a3['systole_err_ms_mean'])} & {_fmt(a3['diastole_err_ms_mean'])} & "
                 f"{_fmt(a3['mean_abs_boundary_err_ms'])} \\\\")
    lines.append('\\bottomrule')
    lines.append('\\end{tabular}')
    lines.append('\\end{table}')

    table_tex = '\n'.join(lines)
    table_path = os.path.join(OUT_DIR, 'exp3_boundary_table.tex')
    with open(table_path, 'w') as fh:
        fh.write(table_tex + '\n')
    print(f'Saved LaTeX table -> {table_path}\n')
    print(table_tex)
else:
    print('No results. Skipping LaTeX table.')

## Cell 8 - IEEE double-column figure

The left panel shows one example PCG with the three sets of phase boundaries
overlaid. True boundaries are solid, reconstructed boundaries are dashed, and
PCG-only boundaries are dotted. The gold phases are shaded for context. The
right panel shows the mean boundary error per arm.

In [ ]:
def _choose_example():
    """Pick a clean example segment. Prefer the named chunk, else best correlation."""
    eligible = [r for r in records
                if r.get('n_gold_cycles', 0) >= 3
                and len(r.get('recon_anchors', [])) >= 3
                and len(r.get('s1_anchors', [])) >= 3]
    if not eligible:
        return None
    for r in eligible:
        if r['key'] == PREFERRED_EXAMPLE:
            return r
    eligible.sort(key=lambda r: (r.get('corr') if r.get('corr') == r.get('corr') else -1),
                  reverse=True)
    return eligible[0]


def _arm_boundary_times(anchors, win_start, win_end):
    """Boundary sample positions inside a window for one arm."""
    anchors = np.sort(np.asarray(anchors, dtype=int))
    lens = _next_interval_map(anchors)
    pts = []
    for a in anchors:
        la = lens.get(int(a))
        if la is None:
            continue
        for phase in PHASE_ORDER:
            b = a + PHASE_FRACTIONS[phase] * la
            if win_start <= b <= win_end:
                pts.append(b)
    return pts


if results:
    from matplotlib.lines import Line2D

    ex = _choose_example()
    fig, (axA, axB) = IEEE.new_fig('double', height=3.0, nrows=1, ncols=2,
                                   gridspec_kw={'width_ratios': [2.6, 1.0]})

    # ---- Panel A: example PCG with three sets of phase boundaries ----
    if ex is not None:
        pcg = ex['pcg_norm']
        gold = np.sort(ex['gold_anchors'])
        # Window over two gold cycles near the middle of the segment.
        mid = max(0, len(gold) // 2 - 1)
        mid = min(mid, len(gold) - 3)
        t0, t2 = int(gold[mid]), int(gold[mid + 2])
        pad = int(0.06 * FS)
        win_start = max(0, t0 - pad)
        win_end = min(len(pcg) - 1, t2 + pad)

        tt = (np.arange(win_start, win_end) - win_start) / FS
        axA.plot(tt, pcg[win_start:win_end], color=IEEE.COLORS['pcg'],
                 linewidth=0.8, zorder=2)

        # Shade gold phases for the two cycles and label them once.
        phase_colors = {'S1': IEEE.COLORS['truth'], 'systole': IEEE.COLORS['proposed'],
                        'S2': IEEE.COLORS['wgan'], 'diastole': IEEE.COLORS['muted']}
        span = {'S1': (0.00, 0.15), 'systole': (0.15, 0.50),
                'S2': (0.50, 0.60), 'diastole': (0.60, 1.00)}
        y0, y1 = float(np.min(pcg[win_start:win_end])), float(np.max(pcg[win_start:win_end]))
        yr = y1 - y0
        for c in (mid, mid + 1):
            g = int(gold[c])
            lg = int(gold[c + 1] - gold[c])
            for phase in PHASE_ORDER:
                f_lo, f_hi = span[phase]
                x_lo = (g + f_lo * lg - win_start) / FS
                x_hi = (g + f_hi * lg - win_start) / FS
                axA.axvspan(x_lo, x_hi, color=phase_colors[phase], alpha=0.12, zorder=1)
                if c == mid:
                    axA.text((x_lo + x_hi) / 2.0, y1 + 0.06 * yr, phase,
                             ha='center', va='bottom', fontsize=6.0,
                             color=phase_colors[phase], fontweight='bold')

        # Overlay each arm's phase boundaries: color and linestyle both encode the arm.
        arm_style = [
            ('gold', ex['gold_anchors'], 'black', '-', 1.0),
            ('recon', ex['recon_anchors'], IEEE.COLORS['proposed'], '--', 1.0),
            ('pcg', ex['s1_anchors'], IEEE.COLORS['cnn'], ':', 1.1),
        ]
        for _, anchors, color, ls, lw in arm_style:
            for b in _arm_boundary_times(anchors, win_start, win_end):
                axA.axvline((b - win_start) / FS, color=color, linestyle=ls,
                            linewidth=lw, alpha=0.9, zorder=3)

        axA.set_ylim(y0 - 0.08 * yr, y1 + 0.22 * yr)
        axA.set_xlim(tt[0], tt[-1])
        IEEE.finalize_axes(axA, xlabel='Time (s)', ylabel='PCG amplitude (a.u.)',
                           title=f"Phase boundaries: {ex['key']}")
        legend_handles = [
            Line2D([0], [0], color='black', linestyle='-', linewidth=1.1,
                   label='True ECG (gold)'),
            Line2D([0], [0], color=IEEE.COLORS['proposed'], linestyle='--', linewidth=1.1,
                   label='Reconstructed ECG'),
            Line2D([0], [0], color=IEEE.COLORS['cnn'], linestyle=':', linewidth=1.3,
                   label='PCG-only S1'),
        ]
        axA.legend(handles=legend_handles, loc='lower right', fontsize=6.5,
                   framealpha=0.92, ncol=1)
    else:
        axA.text(0.5, 0.5, 'No clean example segment', ha='center', va='center',
                 transform=axA.transAxes)
        IEEE.finalize_axes(axA, xlabel='Time (s)', ylabel='PCG amplitude (a.u.)')

    # ---- Panel B: mean boundary error per arm ----
    a2 = results['arms']['arm2_reconstructed_ecg']
    a3 = results['arms']['arm3_pcg_only_s1']
    bar_labels = ['True ECG\n(ref)', 'Recon.\nECG', 'PCG-only\nS1']
    bar_vals = [0.0, a2['mean_abs_boundary_err_ms'], a3['mean_abs_boundary_err_ms']]
    bar_err = [0.0, a2['boundary_err_ms_std'], a3['boundary_err_ms_std']]
    bar_colors = [IEEE.COLORS['truth'], IEEE.COLORS['proposed'], IEEE.COLORS['cnn']]

    xpos = np.arange(len(bar_labels))
    axB.bar(xpos, bar_vals, yerr=bar_err, width=0.66, color=bar_colors,
            edgecolor='black', linewidth=0.7, capsize=2.5, error_kw={'elinewidth': 0.8})
    ymax = max([v + e for v, e in zip(bar_vals, bar_err)] + [1.0])
    for x, v, e in zip(xpos, bar_vals, bar_err):
        axB.text(x, v + (e if e == e else 0.0) + 0.03 * ymax,
                 f'{v:.0f}', ha='center', va='bottom', fontsize=6.5, fontweight='bold')
    axB.set_ylim(0, ymax * 1.18)
    axB.set_xticks(xpos)
    axB.set_xticklabels(bar_labels)
    IEEE.finalize_axes(axB, xlabel='Reconstruction arm',
                       ylabel='Mean boundary error (ms)')

    IEEE.save_fig(fig, OUT_DIR, 'exp3_boundary_figure')
    print('Figure saved (PDF + PNG).')
else:
    print('No results. Skipping figure.')

## Cell 9 - manifest

The output folder now holds the results JSON, the LaTeX table, and the figure.
The reconstruction cache is shared with exp1.

In [ ]:
if results:
    print('exp3 outputs written to:', OUT_DIR)
    for name in ('exp3_boundary_results.json', 'exp3_boundary_table.tex',
                 'exp3_boundary_figure.pdf', 'exp3_boundary_figure.png'):
        p = os.path.join(OUT_DIR, name)
        print(f'  {"[ok] " if os.path.exists(p) else "[missing] "}{p}')
    print('\nThis is the same-dataset true-versus-reconstructed comparison the')
    print('reviewer requested. EPHNOGRAM supplies paired ECG, so arm 1 is a true')
    print('gold reference on the same signals.')
else:
    print('No outputs produced. Verify config_paths data and checkpoint paths.')